# Sales Performance Dashboard: Data Cleaning & EDA
**Domain:** Retail & E-commerce | **Tools:** Python (Pandas, NumPy, Matplotlib)

Is notebook mein hum 5 kaam karte hain:
1. Raw data load + explore
2. Data quality checks (nulls, duplicates, negative sales, invalid dates, inconsistent categories)
3. Cleaning rules apply karna (har rule ka count log hota hai)
4. Cleaned data export (CSV for Power BI + normalized tables for Oracle)
5. EDA: top products, category profit, monthly trend, regional performance

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50, "display.width", 160)
ROOT = Path.cwd()
if ROOT.name in ("notebooks", "scripts"):
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "Superstore_raw.csv"
CLEAN_DIR = ROOT / "data" / "cleaned"
IMG = ROOT / "images"
CLEAN_DIR.mkdir(exist_ok=True, parents=True)
IMG.mkdir(exist_ok=True, parents=True)
TODAY = pd.Timestamp("2026-11-26")  # dates after this are impossible (future)

## 1. Load & Explore

In [2]:
# encoding='latin-1' zaroori hai kyunki Kaggle Superstore file UTF-8 nahi hoti
df = pd.read_csv(RAW, encoding="latin-1")
df.columns = df.columns.str.strip()
print("Shape:", df.shape)
df.head()

Shape: (10054, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Payment Mode
0,5474,IN-2025-002500,06/05/2025,06/07/2025,First Class,CU-00647,Lucas Smith,Home Office,United States,Minneapolis,Minnesota,Central,TEC-MA-10000041,TECHNOLOGY,Machines,Thermal Receipt Printer,428.41,3,0.2,-103.14,UPI
1,941,IN-2023-000433,06/28/2023,07/02/2023,Standard Class,CU-00029,Charlotte Davis,Home Office,United States,Detroit,Michigan,Central,FUR-CH-10000003,Furniture,Chairs,Task Chair with Arms,483.62,3,0.0,60.30,UPI
2,1504,IN-2024-000695,08/14/2024,08/18/2024,Standard Class,CU-00699,James Garcia,Consumer,United States,Richmond,Virginia,East,OFF-ST-10000024,Office Supplies,Storage,Lockable File Cabinet,684.10,4,0.1,85.54,UPI
3,5850,IN-2024-002668,06/17/2024,06/24/2024,Standard Class,CU-00777,Logan Thompson,Corporate,United States,Los Angeles,California,West,FUR-TA-10000005,Furniture,Tables,Oak Conference Table,479.85,1,0.0,33.69,Debit Card
4,1605,IN-2023-000740,08/23/2023,08/24/2023,First Class,CU-00652,Benjamin Mehta,Corporate,United States,San Francisco,California,West,TEC-CO-10000038,Technology,Copiers,Laser Copier Workstation,4642.28,3,0.0,2314.92,UPI


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10054 entries, 0 to 10053
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10054 non-null  int64  
 1   Order ID       10054 non-null  str    
 2   Order Date     10053 non-null  str    
 3   Ship Date      10054 non-null  str    
 4   Ship Mode      10054 non-null  str    
 5   Customer ID    10054 non-null  str    
 6   Customer Name  10019 non-null  str    
 7   Segment        10024 non-null  str    
 8   Country        10054 non-null  str    
 9   City           10054 non-null  str    
 10  State          10054 non-null  str    
 11  Region         10034 non-null  str    
 12  Product ID     10054 non-null  str    
 13  Category       10054 non-null  str    
 14  Sub-Category   10054 non-null  str    
 15  Product Name   10054 non-null  str    
 16  Sales          10054 non-null  float64
 17  Quantity       10054 non-null  int64  
 18  Discount       10

In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,10054.0,NaN,NaN,NaN,4996.056594,2883.224307,1.0,2501.25,4992.5,7491.75,9994.0
Order ID,10054,4500,IN-2024-003247,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,10053,1371,11/23/2025,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,10054,1346,11/29/2025,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,10054,4,Standard Class,5969,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,10054,800,CU-00237,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,10019,428,Lucas Davis,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,10024,3,Consumer,5179,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,10054,1,United States,10054,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,10054,24,Boston,971,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Data Quality Audit (cleaning se PEHLE)

In [5]:
audit = {}
audit["rows_raw"] = len(df)
audit["exact_duplicate_rows"] = int(df.duplicated().sum())
audit["null_cells_by_column"] = {k: int(v) for k, v in df.isna().sum().items() if v > 0}
audit["negative_sales_rows"] = int((df["Sales"] < 0).sum())
audit["zero_sales_rows"] = int((df["Sales"] == 0).sum())
parsed = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")
audit["invalid_order_dates"] = int(parsed.isna().sum())
audit["future_order_dates"] = int((parsed > TODAY).sum())
audit["category_variants_raw"] = sorted(df["Category"].dropna().unique().tolist())
audit["region_variants_raw"] = sorted(df["Region"].dropna().unique().tolist())
for k, v in audit.items():
    print(f"{k}: {v}")

rows_raw: 10054
exact_duplicate_rows: 60
null_cells_by_column: {'Order Date': 1, 'Customer Name': 35, 'Segment': 30, 'Region': 20, 'Discount': 25, 'Profit': 8, 'Payment Mode': 40}
negative_sales_rows: 14
zero_sales_rows: 4
invalid_order_dates: 5
future_order_dates: 1
category_variants_raw: ['FURNITURE', 'Furniture', 'OFFICE SUPPLIES', 'Office Supplies', 'TECHNOLOGY', 'Technology', 'furniture ', 'office supplies ', 'technology ']
region_variants_raw: ['Central', 'East', 'South', 'West', 'central', 'east', 'south', 'west']


## 3. Cleaning (har step ka log)

In [6]:
log = []
def step(name, before, after):
    log.append({"step": name, "rows_before": before, "rows_after": after, "rows_affected": before - after})
    print(f"{name:<55} {before:>6} -> {after:>6}")

# 3.1 Exact duplicates
n = len(df)
df = df.drop_duplicates().copy()
step("Remove exact duplicate rows", n, len(df))

# 3.2 Standardise text columns (spaces + casing)
for c in ["Category", "Sub-Category", "Region", "Segment", "Customer Name", "Payment Mode", "Ship Mode", "State", "City"]:
    df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
for c in ["Category", "Sub-Category", "Region"]:
    df[c] = df[c].str.title()
print("Categories now:", sorted(df["Category"].dropna().unique()))
print("Regions now   :", sorted(df["Region"].dropna().unique()))

# 3.3 Category must agree with Sub-Category (one sub-category belongs to ONE category)
sub_to_cat = df.dropna(subset=["Category"]).groupby("Sub-Category")["Category"].agg(lambda s: s.mode().iat[0])
mismatch = (df["Category"] != df["Sub-Category"].map(sub_to_cat)).sum()
print("Category/Sub-Category mismatches:", int(mismatch))
df["Category"] = df["Sub-Category"].map(sub_to_cat)

# 3.4 Dates: parse strictly; recover from other lines of the same order, else drop
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y", errors="coerce")
df.loc[df["Order Date"] > TODAY, "Order Date"] = pd.NaT
bad_dates = df["Order Date"].isna().sum()
sibling_date = df.groupby("Order ID")["Order Date"].transform("min")  # min ignores NaT
df["Order Date"] = df["Order Date"].fillna(sibling_date)
print(f"Invalid dates: {bad_dates}, recovered from same order: {bad_dates - df['Order Date'].isna().sum()}")
n = len(df)
df = df.dropna(subset=["Order Date"])
step("Drop rows with unrecoverable invalid Order Date", n, len(df))

# 3.5 Negative / zero sales are data-entry errors (returns are not in this feed) -> drop
n = len(df)
df = df[df["Sales"] > 0]
step("Drop rows with negative or zero Sales", n, len(df))

# 3.6 Missing values
# Customer attributes come from customer master (same Customer ID)
cust_master = df.dropna(subset=["Customer Name", "Segment"]).groupby("Customer ID").agg(
    name=("Customer Name", lambda s: s.mode().iat[0]), seg=("Segment", lambda s: s.mode().iat[0]))
df["Customer Name"] = df["Customer Name"].fillna(df["Customer ID"].map(cust_master["name"]))
df["Segment"] = df["Segment"].fillna(df["Customer ID"].map(cust_master["seg"]))
# Region from State
state_region = df.dropna(subset=["Region"]).groupby("State")["Region"].agg(lambda s: s.mode().iat[0])
df["Region"] = df["Region"].fillna(df["State"].map(state_region))
# Payment mode from other lines of same order, else 'Unknown'
order_pay = df.dropna(subset=["Payment Mode"]).groupby("Order ID")["Payment Mode"].agg(lambda s: s.mode().iat[0])
df["Payment Mode"] = df["Payment Mode"].fillna(df["Order ID"].map(order_pay)).fillna("Unknown")
# Customer with only one line and a missing name cannot be recovered -> keep the row, label it
df["Customer Name"] = df["Customer Name"].fillna("Unknown Customer")
# Discount missing -> 0 (no discount recorded)
df["Discount"] = df["Discount"].fillna(0.0)
# Profit missing cannot be guessed reliably -> drop
n = len(df)
df = df.dropna(subset=["Profit"])
step("Drop rows with missing Profit", n, len(df))
print("Remaining nulls:", int(df.isna().sum().sum()))

# 3.7 Logical checks
print("Rows where Ship Date < Order Date:", int((df["Ship Date"] < df["Order Date"]).sum()))
n = len(df)
df = df[(df["Quantity"] > 0) & (df["Discount"].between(0, 1))]
step("Drop rows with invalid Quantity/Discount", n, len(df))

# 3.8 Feature engineering
df["Year"] = df["Order Date"].dt.year
df["Month Start"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Month Name"] = df["Order Date"].dt.strftime("%b")
df["Month Number"] = df["Order Date"].dt.month
df["Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str)
df["Ship Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin %"] = np.round(df["Profit"] / df["Sales"] * 100, 2)
df["Sales"] = df["Sales"].round(2)
df["Profit"] = df["Profit"].round(2)
df = df.sort_values(["Order Date", "Order ID", "Row ID"]).reset_index(drop=True)

# 3.9 Order-level consistency (one order = one date/customer/payment/region)
for c in ["Order Date", "Customer ID", "Payment Mode", "Region", "Ship Mode"]:
    bad = (df.groupby("Order ID")[c].nunique() > 1).sum()
    print(f"orders with inconsistent {c}: {bad}")

Remove exact duplicate rows                              10054 ->   9994
Categories now: ['Furniture', 'Office Supplies', 'Technology']
Regions now   : ['Central', 'East', 'South', 'West']
Category/Sub-Category mismatches: 0
Invalid dates: 6, recovered from same order: 4
Drop rows with unrecoverable invalid Order Date           9994 ->   9992
Drop rows with negative or zero Sales                     9992 ->   9974
Drop rows with missing Profit                             9974 ->   9966
Remaining nulls: 0
Rows where Ship Date < Order Date: 0
Drop rows with invalid Quantity/Discount                  9966 ->   9966
orders with inconsistent Order Date: 0
orders with inconsistent Customer ID: 0
orders with inconsistent Payment Mode: 0
orders with inconsistent Region: 0
orders with inconsistent Ship Mode: 0


## 4. Export cleaned data

In [7]:
df.to_csv(CLEAN_DIR / "sales_cleaned.csv", index=False)

# Normalised tables for Oracle (ISO dates so TO_DATE(...,'YYYY-MM-DD') works)
customers = (df.groupby("Customer ID").agg(customer_name=("Customer Name", "first"), segment=("Segment", "first"))
               .reset_index().rename(columns={"Customer ID": "customer_id"}))
products = (df.groupby("Product ID").agg(product_name=("Product Name", "first"), category=("Category", "first"),
                                         sub_category=("Sub-Category", "first"))
              .reset_index().rename(columns={"Product ID": "product_id"}))
orders = (df.groupby("Order ID").agg(order_date=("Order Date", "min"), ship_date=("Ship Date", "min"),
                                      ship_mode=("Ship Mode", "first"), customer_id=("Customer ID", "first"),
                                      region=("Region", "first"), state=("State", "first"), city=("City", "first"),
                                      payment_mode=("Payment Mode", "first")).reset_index()
            .rename(columns={"Order ID": "order_id"}))
for c in ["order_date", "ship_date"]:
    orders[c] = orders[c].dt.strftime("%Y-%m-%d")
items = df[["Row ID", "Order ID", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].copy()
items.columns = ["row_id", "order_id", "product_id", "sales", "quantity", "discount", "profit"]
for name, t in [("customers", customers), ("products", products), ("orders", orders), ("order_items", items)]:
    t.to_csv(CLEAN_DIR / f"{name}.csv", index=False)
    print(f"{name:<12} rows={len(t)}")

pd.DataFrame(log).to_csv(CLEAN_DIR / "cleaning_log.csv", index=False)

customers    rows=800
products     rows=42
orders       rows=4497
order_items  rows=9966


## 5. KPIs (yahi numbers Oracle SQL aur Power BI se match hone chahiye)

In [8]:
kpi = {
    "total_sales": round(float(df["Sales"].sum()), 2),
    "total_profit": round(float(df["Profit"].sum()), 2),
    "total_orders": int(df["Order ID"].nunique()),
    "total_quantity": int(df["Quantity"].sum()),
    "profit_margin_pct": round(float(df["Profit"].sum() / df["Sales"].sum() * 100), 2),
    "avg_order_value": round(float(df["Sales"].sum() / df["Order ID"].nunique()), 2),
    "line_items": int(len(df)),
}
print(json.dumps(kpi, indent=2))

{
  "total_sales": 5408541.6,
  "total_profit": 720415.35,
  "total_orders": 4497,
  "total_quantity": 29913,
  "profit_margin_pct": 13.32,
  "avg_order_value": 1202.7,
  "line_items": 9966
}


## 6. Exploratory Data Analysis

In [9]:
# 6.1 Top 10 products by sales
top_products = (df.groupby("Product Name").agg(sales=("Sales", "sum"), profit=("Profit", "sum"), qty=("Quantity", "sum"))
                  .sort_values("sales", ascending=False).head(10).round(2))
print(top_products)
kpi["top10_products"] = [{"rank": i + 1, "product": p, "sales": float(r.sales)} for i, (p, r) in enumerate(top_products.iterrows())]

fig, ax = plt.subplots(figsize=(9, 5))
top_products["sales"].sort_values().plot.barh(ax=ax, color="#2563EB")
ax.set_title("Top 10 Products by Sales"); ax.set_xlabel("Sales ($)")
plt.tight_layout(); plt.savefig(IMG / "top10_products.png", dpi=130); plt.close()

                                sales     profit  qty
Product Name                                         
Laser Copier Workstation   1036962.57  299571.06  705
Compact Desktop Copier      505391.16  143836.47  728
Smartphone Rugged Edition   343757.27   33010.36  806
Oak Conference Table        336868.70  -10420.43  729
Smartphone 128GB Black      322610.50   35985.06  665
Executive Leather Chair     271748.26   35269.26  806
Adjustable Standing Desk    240743.12   -4537.84  650
Round Meeting Table         213962.51   -4787.25  718
Corner Bookcase Unit        163757.92    2270.86  762
Label Printer Pro           158509.77     331.63  673


In [10]:
# 6.2 Profitable categories / sub-categories
cat = df.groupby("Category").agg(sales=("Sales", "sum"), profit=("Profit", "sum"))
cat["margin_%"] = (cat["profit"] / cat["sales"] * 100).round(2)
print(cat.round(2))
sub = df.groupby("Sub-Category").agg(sales=("Sales", "sum"), profit=("Profit", "sum"))
sub["margin_%"] = (sub["profit"] / sub["sales"] * 100).round(2)
print(sub.sort_values("profit").round(2))
kpi["category"] = cat.round(2).reset_index().to_dict("records")
kpi["subcategory"] = sub.round(2).reset_index().to_dict("records")

fig, ax = plt.subplots(figsize=(9, 5))
sub["profit"].sort_values().plot.barh(ax=ax, color=["#DC2626" if v < 0 else "#059669" for v in sub["profit"].sort_values()])
ax.set_title("Profit by Sub-Category"); ax.set_xlabel("Profit ($)")
plt.tight_layout(); plt.savefig(IMG / "profit_by_subcategory.png", dpi=130); plt.close()

                      sales     profit  margin_%
Category                                        
Furniture        2020838.57   85276.06      4.22
Office Supplies   427352.55   66718.23     15.61
Technology       2960350.48  568421.06     19.20
                   sales     profit  margin_%
Sub-Category                                 
Tables         889365.48  -23151.75     -2.60
Machines       396169.09    -500.52     -0.13
Bookcases      396261.48    4204.57      1.06
Art             17058.19    4630.09     27.14
Binders         28162.42    8930.85     31.71
Storage        185632.66   10361.36      5.58
Paper           39229.11   12842.72     32.74
Furnishings    107016.05   22908.63     21.41
Appliances     157270.17   29953.21     19.05
Accessories    204718.61   41671.32     20.36
Chairs         628195.56   81314.61     12.94
Phones         817109.05   83842.73     10.26
Copiers       1542353.73  443407.53     28.75


In [11]:
# 6.3 Monthly trend + MoM growth
monthly = df.groupby("Month Start").agg(sales=("Sales", "sum"), profit=("Profit", "sum"), orders=("Order ID", "nunique"))
monthly["mom_growth_%"] = (monthly["sales"].pct_change() * 100).round(2)
print(monthly.round(2).tail(12))
kpi["yearly"] = (df.groupby("Year").agg(sales=("Sales", "sum"), profit=("Profit", "sum"), orders=("Order ID", "nunique"))
                   .round(2).reset_index().to_dict("records"))
kpi["monthly_sales_first6"] = monthly["sales"].round(2).head(6).reset_index().astype({"Month Start": str}).to_dict("records")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(monthly.index, monthly["sales"], color="#2563EB", lw=2, marker="o", ms=3)
ax.set_title("Monthly Sales Trend"); ax.set_ylabel("Sales ($)"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(IMG / "monthly_sales_trend.png", dpi=130); plt.close()

                 sales    profit  orders  mom_growth_%
Month Start                                           
2025-01-01    68711.47   9714.10      65        -65.79
2025-02-01   106118.73  14792.50      80         54.44
2025-03-01    96770.67   9278.82      93         -8.81
2025-04-01   106104.00  14912.15     100          9.64
2025-05-01   115228.20  15467.65      94          8.60
2025-06-01   116346.36  10477.94      98          0.97
2025-07-01    99976.84  11498.97      84        -14.07
2025-08-01   146669.15  24326.66     106         46.70
2025-09-01   144606.38  17144.33     133         -1.41
2025-10-01    82941.93   5846.35      98        -42.64
2025-11-01   232461.23  45530.41     161        180.27
2025-12-01   245287.23  35049.67     184          5.52


In [12]:
# 6.4 Regional performance
reg = df.groupby("Region").agg(sales=("Sales", "sum"), profit=("Profit", "sum"), orders=("Order ID", "nunique"))
reg["margin_%"] = (reg["profit"] / reg["sales"] * 100).round(2)
print(reg.sort_values("sales", ascending=False).round(2))
kpi["region"] = reg.round(2).reset_index().to_dict("records")

fig, ax = plt.subplots(figsize=(7, 4.5))
reg["sales"].sort_values().plot.barh(ax=ax, color="#7C3AED")
ax.set_title("Sales by Region"); ax.set_xlabel("Sales ($)")
plt.tight_layout(); plt.savefig(IMG / "sales_by_region.png", dpi=130); plt.close()

              sales     profit  orders  margin_%
Region                                          
West     1714998.90  286112.79    1398     16.68
East     1578776.59  251574.07    1326     15.93
Central  1285032.71   91911.06    1101      7.15
South     829733.40   90817.43     672     10.95


In [13]:
# 6.5 Discount impact on profit (important business insight)
bins = pd.cut(df["Discount"], [-0.01, 0, 0.10, 0.20, 0.30, 1.0], labels=["0%", "1-10%", "11-20%", "21-30%", ">30%"])
disc = df.groupby(bins, observed=True).agg(sales=("Sales", "sum"), profit=("Profit", "sum"), lines=("Row ID", "count"))
disc["margin_%"] = (disc["profit"] / disc["sales"] * 100).round(2)
print(disc.round(2))
kpi["discount_bands"] = disc.round(2).reset_index().rename(columns={"Discount": "band"}).astype({"band": str}).to_dict("records")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(disc.index.astype(str), disc["margin_%"], color=["#059669" if v > 0 else "#DC2626" for v in disc["margin_%"]])
ax.axhline(0, color="black", lw=.8); ax.set_title("Profit Margin % by Discount Band"); ax.set_ylabel("Margin %")
plt.tight_layout(); plt.savefig(IMG / "discount_vs_margin.png", dpi=130); plt.close()

               sales     profit  lines  margin_%
Discount                                        
0%        2858187.12  647161.13   4709     22.64
1-10%      977652.37  133162.67   1805     13.62
11-20%    1031855.75   27304.87   2052      2.65
21-30%     315627.66  -28209.52    722     -8.94
>30%       225218.70  -59003.80    678    -26.20


In [14]:
# 6.6 Segment & payment mode & profit-vs-sales scatter
seg = df.groupby("Segment").agg(sales=("Sales", "sum"), profit=("Profit", "sum"))
pay = df.groupby("Payment Mode").agg(sales=("Sales", "sum"), orders=("Order ID", "nunique"))
print(seg.round(2)); print(pay.round(2))
kpi["segment"] = seg.round(2).reset_index().to_dict("records")
kpi["payment_mode"] = pay.round(2).reset_index().to_dict("records")

sp = df.groupby("Product Name").agg(sales=("Sales", "sum"), profit=("Profit", "sum"))
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sp["sales"], sp["profit"], c=["#DC2626" if p < 0 else "#2563EB" for p in sp["profit"]], alpha=.8)
ax.axhline(0, color="black", lw=.8); ax.set_xlabel("Sales ($)"); ax.set_ylabel("Profit ($)"); ax.set_title("Profit vs Sales by Product")
plt.tight_layout(); plt.savefig(IMG / "profit_vs_sales.png", dpi=130); plt.close()
loss = sp[sp["profit"] < 0].sort_values("profit")
print("Loss-making products:", len(loss)); print(loss.round(2))
kpi["loss_products"] = loss.round(2).reset_index().to_dict("records")

                  sales     profit
Segment                           
Consumer     2704998.56  359228.28
Corporate    1625360.73  216017.01
Home Office  1078182.31  145170.06
                       sales  orders
Payment Mode                        
Cash on Delivery   563743.17     482
Credit Card       1595579.55    1362
Debit Card        1124042.82     903
Net Banking        512232.92     454
PayPal             598013.92     494
UPI               1011057.01     794
Unknown              3872.21       8
Loss-making products: 6
                              sales    profit
Product Name                                 
Oak Conference Table      336868.70 -10420.43
Round Meeting Table       213962.51  -4787.25
Adjustable Standing Desk  240743.12  -4537.84
Folding Training Table     97791.15  -3406.23
Heavy Duty Laminator       97149.14  -1039.94
5-Shelf Wooden Bookcase   133273.21    -86.93


In [15]:
# Save audit + KPI reference (acceptance criteria ke liye)
(ROOT / "docs").mkdir(exist_ok=True)
with open(ROOT / "docs" / "expected_kpis.json", "w") as f:
    json.dump({"audit_before_cleaning": audit, "cleaning_log": log, "kpis": kpi}, f, indent=2, default=str)
print("Done. Cleaned rows:", len(df))

Done. Cleaned rows: 9966
